# Single-Cell RNA-seq Analysis: GSE184880

This notebook implements a full scRNA-seq analysis pipeline using [Scanpy](https://scanpy.readthedocs.io/), applied to the publicly available dataset **GSE184880** (colorectal cancer, Cancer vs Normal samples).

**Pipeline overview:**
1. Data import & QC
2. Cell cycle scoring & regression
3. Dimensionality reduction & clustering (PCA → UMAP → Leiden)
4. Cell type annotation
5. Differential expression (Tumour vs Immune; Cancer vs Normal)
6. Immune subclustering & functional state scoring
7. Pathway enrichment (GSEA / Enrichr)

> **Before running:** set `base_dir` in the first code cell to the folder containing your 10x Genomics MTX sample directories.

---
*Analysis by [Jaysmita Chaliha](https://github.com/jaysmitachaliha) | Dataset: [GSE184880](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE184880)*

## 0. Data Setup

Before running the analysis, download the raw data from GEO and organise it into the expected directory structure. Run the cell below **once** to set up the `data/` folder. You can skip this if you have already done so.

In [ ]:
# Data setup — run once to organise downloaded GEO files into per-sample directories
# Download GSE184880 supplementary files from:
# https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE184880
# Place all downloaded .gz files into a single folder and set raw_dir below.

import os
import shutil

raw_dir = "/path/to/downloaded/GSE184880/files"  # <-- update this
base_dir = "data"

for fname in os.listdir(raw_dir):
    if not fname.endswith(".gz"):
        continue

    # e.g. "GSM5599220_Norm1.barcodes.tsv.gz" → sample = "GSM5599220_Norm1"
    sample = fname.split(".")[0]
    sample_dir = os.path.join(base_dir, sample)
    os.makedirs(sample_dir, exist_ok=True)

    if "barcodes" in fname:
        dest = os.path.join(sample_dir, "barcodes.tsv.gz")
    elif "genes" in fname:
        dest = os.path.join(sample_dir, "features.tsv.gz")  # scanpy expects "features"
    elif "matrix" in fname:
        dest = os.path.join(sample_dir, "matrix.mtx.gz")
    else:
        continue

    shutil.copy(os.path.join(raw_dir, fname), dest)
    print(f"{fname} → {dest}")

print("\nData setup complete.")

## 1. Data Import & Preprocessing

Samples were obtained from the Gene Expression Omnibus (GEO) under accession [GSE184880](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE184880). This dataset comprises colorectal tumour and matched adjacent normal tissue from 12 donors (5 Normal, 7 Cancer), profiled using the 10x Genomics Chromium platform.

Each sample is provided as a standard 10x MTX directory (matrix, barcodes, features). Samples are loaded individually, labelled by condition, and concatenated into a single AnnData object. Low-quality cells are removed based on three criteria: minimum gene detection (>300 genes), maximum gene detection (<6,000 genes, to exclude doublets), and mitochondrial read fraction (<15%, to exclude damaged cells). Expression values are then library-size normalised to 10,000 counts per cell and log1p-transformed. The 2,000 most highly variable genes (HVGs) are selected using Seurat's dispersion-based method, applied in a batch-aware manner to avoid sample-specific bias.

In [ ]:
# Import libraries & settings

import igraph
import leidenalg
import scanpy as sc
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import gseapy as gp
import importlib.metadata
importlib.metadata.version("scanpy")

sc.settings.verbosity = 3
sc.set_figure_params(figsize=(5,5))

In [ ]:
# Load and merge samples

base_dir = "data"  # <-- set this to your local data directory

sample_dirs = sorted([
    d for d in os.listdir(base_dir)
    if os.path.isdir(os.path.join(base_dir, d))
])

adatas = []

for d in sample_dirs:
    ad = sc.read_10x_mtx(
        os.path.join(base_dir, d),
        var_names="gene_symbols",
        cache=True
    )
    ad.obs["sample"] = d
    ad.obs["condition"] = "Normal" if "Norm" in d else "Cancer"
    adatas.append(ad)
    
adata = adatas[0].concatenate(
    adatas[1:],
    batch_key="sample_id",
    batch_categories=sample_dirs
)

n_cells, n_genes = adata.n_obs, adata.n_vars

print(f"Total cells: {n_cells}")
print(f"Total genes: {n_genes}")
adata.obs["sample"].value_counts()
adata.obs["condition"].value_counts()


In [ ]:
# Quality control

adata.var["mt"] = adata.var_names.str.startswith("MT-")

sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt"],
    inplace=True
)

adata = adata[
    (adata.obs.n_genes_by_counts > 300) &
    (adata.obs.n_genes_by_counts < 6000) &
    (adata.obs.pct_counts_mt < 15)
].copy()

n_cells_post_qc = adata.n_obs
print(f"Cells after QC and preprocessing: {n_cells_post_qc}")

In [ ]:
# Normalization & log transform

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

adata.raw = adata.copy()

print(f"Total genes (pre-HVG): {adata.raw.n_vars}")

In [ ]:
# Highly variable genes

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    flavor="seurat",
    batch_key="sample"
)

adata = adata[:, adata.var.highly_variable].copy()

n_genes_hvg = adata.n_vars
print(f"Genes after preprocessing (HVGs): {n_genes_hvg}")

print(
    f"After quality control, {adata.n_obs} cells "
    f"and {adata.raw.n_vars} genes were retained; "
    f"{adata.n_vars} highly variable genes were used "
    f"for downstream analyses."
)

cells_per_sample = adata.obs["sample"].value_counts()
print(cells_per_sample)


qc_summary = (
    adata.obs
    .groupby("sample")
    .size()
    .rename("cells_post_qc")
)

qc_summary

adata.uns["qc_preproc_summary"] = {
    "cells_post_qc": adata.n_obs,
    "genes_post_qc": adata.raw.n_vars,
    "genes_hvg": adata.n_vars
}

## 2. Cell Cycle Handling & Regression

Cell cycle state is a major source of transcriptional variation in proliferating tissues and can confound clustering if left unaddressed. Each cell is scored for S-phase and G2/M-phase activity using canonical gene signatures from [Tirosh et al. (2016)](https://www.science.org/doi/10.1126/science.aad0501). Scores are then regressed out of the expression matrix, and the data is scaled to unit variance (maximum value capped at 10) to prevent highly expressed genes from dominating downstream analyses.

In [ ]:
# Cell cycle scoring

# Canonical cell cycle genes (Tirosh et al.)
s_genes = [
    "MCM5", "PCNA", "TYMS", "FEN1", "MCM2", "MCM4", "RRM1", "UNG",
    "GINS2", "MCM6", "CDCA7", "DTL", "PRIM1", "UHRF1", "HELLS",
    "RFC2", "RPA2", "NASP", "RAD51AP1", "GMNN", "WDR76", "SLBP",
    "CCNE2", "UBR7", "POLD3", "MSH2", "ATAD2", "RAD51", "RRM2",
    "CDC45", "CDC6", "EXO1", "TIPIN", "DSCC1", "BLM", "CASP8AP2",
    "USP1", "CLSPN", "POLA1", "CHAF1B", "BRIP1", "E2F8"
]

g2m_genes = [
    "HMGB2", "CDK1", "NUSAP1", "UBE2C", "BIRC5", "TPX2", "TOP2A",
    "NDC80", "CKS2", "NUF2", "CKS1B", "MKI67", "TMPO", "CENPF",
    "TACC3", "FAM64A", "SMC4", "CCNB2", "CKAP2L", "CKAP2",
    "AURKB", "BUB1", "KIF11", "ANP32E", "TUBB4B", "GTSE1",
    "KIF20B", "HJURP", "CDC20", "TTK", "CDC25C", "KIF2C",
    "RANGAP1", "NCAPD2", "DLGAP5", "CDCA3", "HMMR", "AURKA",
    "PSRC1", "ANLN", "LBR", "CKAP5", "CENPE", "CTCF",
    "NEK2", "G2E3", "GAS2L3", "CBX5", "CENPA"
]

s_genes = [g for g in s_genes if g in adata.var_names]
g2m_genes = [g for g in g2m_genes if g in adata.var_names]

print("S genes used:", len(s_genes))
print("G2M genes used:", len(g2m_genes))


sc.tl.score_genes_cell_cycle(
    adata,
    s_genes=s_genes,
    g2m_genes=g2m_genes
)

adata.obs[["S_score", "G2M_score", "phase"]].head()

In [ ]:
# Cell cycle regression

sc.pp.regress_out(adata, keys=["S_score", "G2M_score"])

In [ ]:
# Scaling

sc.pp.scale(adata, max_value=10, zero_center=True)

print(adata.obs["phase"].value_counts())

## 3. Dimensionality Reduction & Clustering

Dimensionality reduction follows a standard Scanpy workflow. PCA is performed on the HVG matrix (50 components, ARPACK solver); the number of informative PCs is assessed by variance ratio. A k-nearest neighbour graph is constructed in PCA space (k=15, 30 PCs), and Leiden community detection (resolution=0.3) is applied to identify clusters. UMAP is used for two-dimensional visualisation.

In [ ]:
# PCA

sc.tl.pca(adata, n_comps=50, svd_solver="arpack")

sc.pl.pca_variance_ratio(adata, log=True)

print("Variance explained by first 10 PCs:")
print(adata.uns["pca"]["variance_ratio"][:10])


In [ ]:
# Neighbors graph construction

sc.pp.neighbors(
    adata,
    n_neighbors=15,
    n_pcs=30
)

In [ ]:
# Clustering

sc.tl.leiden(
    adata,
    resolution=0.3,
    key_added="leiden_cc_regressed"
)

adata.obs["leiden_cc_regressed"].value_counts()
print(adata.obs["leiden_cc_regressed"].value_counts())

adata.uns["qc_report"] = {
    "cells_post_qc": adata.n_obs,
    "genes_post_qc": adata.raw.n_vars,
    "genes_hvg": adata.n_vars,
    "median_umis": float(np.median(adata.obs["total_counts"])),
    "median_genes": float(np.median(adata.obs["n_genes_by_counts"])),
    "median_pct_mt": float(np.median(adata.obs["pct_counts_mt"])),
    "cells_per_sample": adata.obs["sample"].value_counts().to_dict(),
    "cells_per_condition": adata.obs["condition"].value_counts().to_dict()
}


In [ ]:
# UMAP

sc.tl.umap(adata, random_state=0)

sc.pl.umap(adata, color="leiden_cc_regressed", title="UMAP embedding of all cells colored by Leiden clusters")
sc.pl.umap(adata, color="condition", title="UMAP embedding colored by experimental condition (Cancer vs Normal)")  


## 4. Cell Identity & Annotation

Cluster identities are assigned in two layers. First, marker genes for each Leiden cluster are identified using a Wilcoxon rank-sum test. Second, clusters are manually annotated by inspecting the expression of canonical lineage markers:

| Cell type | Marker genes |
|-----------|-------------|
| Tumour epithelial | EPCAM, KRT8 |
| T cells | CD3D, TRBC1 |
| B cells | MS4A1 |
| Macrophages | LST1, C1QC |
| NK cells | NKG7, GNLY |
| Fibroblasts | COL1A1 |
| Endothelial | PECAM1, VWF |
| Cycling | MKI67, TOP2A |

Each cell is assigned both a **broad lineage label** (conservative, marker-driven) and a **cluster-state label** (finer functional annotation based on Leiden cluster identity).

In [ ]:
# Marker gene detection

sc.tl.rank_genes_groups(
    adata,
    groupby="leiden_cc_regressed",
    method="wilcoxon"
)

sc.pl.rank_genes_groups_dotplot(
    adata,
    groupby="leiden_cc_regressed",
    n_genes=5,
    title="Dot plot of top marker genes per Leiden cluster"
)

markers = sc.get.rank_genes_groups_df(adata, None)
markers.to_csv(
    "GSE184880_scanpy_leiden_cc_regressed_markers.csv",
    index=False
)

In [ ]:
# Cell-type annotation

# Marker visualization
sc.pl.umap(adata, color=["EPCAM", "KRT8"])                # Tumour epithelial
sc.pl.umap(adata, color=["CD3D", "TRBC1"])                # T cells
sc.pl.umap(adata, color=["MS4A1"])                        # B cells
sc.pl.umap(adata, color=["LST1", "C1QC"])                 # Macrophages
sc.pl.umap(adata, color=["NKG7", "GNLY"])                 # NK cells
sc.pl.umap(adata, color=["COL1A1"])                       # Fibroblasts
sc.pl.umap(adata, color=["PECAM1", "VWF"])                # Endothelial
sc.pl.umap(adata, color=["MKI67", "TOP2A"])               # Cycling

# Broad lineage annotation (robust, conservative)

adata.obs["broad_cell_type"] = "Other"

adata.obs.loc[
    adata.obs["leiden_cc_regressed"].isin(["0","1","2","3"]),
    "broad_cell_type"
] = "Tumour epithelial"

adata.obs.loc[
    adata.obs["leiden_cc_regressed"].isin(["4","5","13"]),
    "broad_cell_type"
] = "T cells"

adata.obs.loc[
    adata.obs["leiden_cc_regressed"].isin(["6","7","14"]),
    "broad_cell_type"
] = "Macrophages"

adata.obs.loc[
    adata.obs["leiden_cc_regressed"] == "8",
    "broad_cell_type"
] = "B cells"

adata.obs.loc[
    adata.obs["leiden_cc_regressed"].isin(["9","15"]),
    "broad_cell_type"
] = "NK cells"

adata.obs.loc[
    adata.obs["leiden_cc_regressed"] == "10",
    "broad_cell_type"
] = "Fibroblasts"

adata.obs.loc[
    adata.obs["leiden_cc_regressed"] == "11",
    "broad_cell_type"
] = "Endothelial"

adata.obs["broad_cell_type"] = adata.obs["broad_cell_type"].astype("category")


# Cluster state annotation (functional / activation states)

cluster_to_state = {
    "0":  "Tumour – baseline",
    "1":  "Tumour – proliferative",
    "2":  "Tumour – EMT-like",
    "3":  "Tumour – stress response",

    "4":  "T cells – naïve",
    "5":  "T cells – cytotoxic",
    "13": "T cells – activated",

    "6":  "Macrophages – C1QC+",
    "7":  "Macrophages – FCGR3A+",
    "14": "Macrophages – inflammatory",

    "8":  "B cells",

    "9":  "NK cells – resting",
    "15": "NK cells – activated",

    "10": "Fibroblasts",
    "11": "Endothelial",

    "12": "Cycling (mixed)",
    "18": "Cycling immune",

    "16": "Dendritic cells",
    "17": "IFN-stimulated immune",

    "19": "Other / ambiguous",
    "20": "Other / ambiguous",
    "21": "Other / ambiguous"
}

adata.obs["cell_type"] = (
    adata.obs["leiden_cc_regressed"]
    .astype(str)
    .map(cluster_to_state)
    .astype("category")
)

adata.obs["cell_type"].value_counts()

# Compare lineage vs state
sc.pl.umap(
    adata,
    color=["broad_cell_type", "cell_type"],
    wspace=0.4,
    frameon=False
)

sc.pl.umap(
    adata,
    color=[
        "EPCAM", "KRT8",        # tumour
        "CD3D", "TRBC1",        # T
        "MS4A1",                # B
        "LST1", "C1QC",         # macrophage
        "NKG7", "GNLY",         # NK
        "PECAM1", "VWF",        # endothelial
        "MKI67", "TOP2A"        # cycling
    ],
    ncols=4
)

sc.pl.umap(adata, color=["broad_cell_type", "cell_type"], title=["Broad cell types", "Cell type states"])

sc.pl.umap(
    adata,
    color="broad_cell_type",
    title="UMAP – Broad cell types",
    frameon=False
)

sc.pl.umap(
    adata,
    color="cell_type",
    title="UMAP – Cluster-defined transcriptional states",
    frameon=False
)


tumour_subsets = adata[adata.obs["cell_type"] == "Tumour epithelial"].copy()
tumour_subsets.write("adata_tumour_only.h5ad")

immune_subsets = adata[
    adata.obs["cell_type"].isin(
        ["T cells", "B cells", "NK cells", "Macrophages"]
    )
].copy()
immune_subsets.write("adata_immune_only.h5ad")

In [ ]:
# Cell cycle scores visualized on UMAP

# Cell cycle phase on UMAP
sc.pl.umap(
    adata,
    color="phase",
    title="UMAP colored by cell cycle phase (G1, S, G2/M)"
)

# Proliferation intensity
sc.pl.umap(
    adata,
    color=["S_score", "G2M_score","phase"],
    title=['S-phase score', 'G2/M-phase score', 'Cell cycle phase']
)

# Cycling vs non-cycling clusters
sc.pl.violin(
    adata,
    keys=["S_score", "G2M_score"],
    groupby="cell_type",
    rotation=90
)

## 5. Differential Expression & Tumour Biology

Two complementary differential expression analyses are performed using the Wilcoxon rank-sum test (FDR corrected).

**Tumour vs Immune:** Tumour epithelial cells are compared against the aggregated immune compartment (T cells, B cells, NK cells, Macrophages) to identify the transcriptional programs that distinguish malignant epithelial cells from infiltrating immune populations.

**Cancer vs Normal (tumour epithelial only):** Within the tumour epithelial subset, Cancer-condition cells are compared against Normal-condition cells to identify genes specifically dysregulated in malignant tissue. Results are visualised as volcano plots, ranked bar charts, heatmaps, and UMAP feature plots.

In [ ]:
# Tumour vs Immune DEG
immune_types = ["T cells", "B cells", "NK cells", "Macrophages"]

adata.obs["broad_type"] = np.where(
    adata.obs["broad_cell_type"] == "Tumour epithelial",
    "Tumour",
    np.where(
        adata.obs["broad_cell_type"].isin(immune_types),
        "Immune",
        "Other"  # <-- use string instead of np.nan
    )
)

adata.obs["broad_type"] = (
    adata.obs["broad_type"]
    .astype("category")
    .cat.remove_unused_categories()
)

# Filter to only Tumour and Immune (exclude "Other")
adata_ti = adata[adata.obs["broad_type"].isin(["Tumour", "Immune"])].copy()
adata_ti.obs["broad_type"] = (
    adata_ti.obs["broad_type"]
    .astype("category")
    .cat.remove_unused_categories()
)

adata_ti.obs["broad_type"].value_counts()

# Make sure it is categorical
adata.obs["broad_type"] = adata.obs["broad_type"].astype("category")

adata.obs["broad_type"].value_counts()


sc.tl.rank_genes_groups(
    adata_ti,
    groupby="broad_type",
    reference="Immune",
    method="wilcoxon"
)

tumour_vs_immune = sc.get.rank_genes_groups_df(
    adata_ti,
    group="Tumour"
)

tumour_vs_immune.to_csv(
    "tumour_vs_immune_DEGs.csv",
    index=False
)

# Visualize top DE genes
tumour_vs_immune.head(10)
top_genes = (
    tumour_vs_immune
    .query("pvals_adj < 0.05")
    .sort_values("logfoldchanges", ascending=False)
    .head(20)["names"]
    .tolist()
)

sc.pl.umap(
    adata_ti,
    color=top_genes,
    ncols=4,
    title=top_genes
)

# Volcano plot (Tumour vs Immune)
import matplotlib.pyplot as plt
import numpy as np

df = tumour_vs_immune.copy()
df["-log10_padj"] = -np.log10(df["pvals_adj"] + 1e-300)

sig = (df["pvals_adj"] < 0.05) & (np.abs(df["logfoldchanges"]) > 0.5)

plt.figure(figsize=(6, 5))
plt.scatter(df.loc[~sig, "logfoldchanges"], df.loc[~sig, "-log10_padj"],
            s=10, color="lightgrey", alpha=0.6)
plt.scatter(df.loc[sig, "logfoldchanges"], df.loc[sig, "-log10_padj"],
            s=12, color="crimson")

plt.axhline(-np.log10(0.05), linestyle="--", color="black")
plt.axvline(0.5, linestyle="--", color="black")
plt.axvline(-0.5, linestyle="--", color="black")

plt.xlabel("log2 fold change (Tumour vs Immune)")
plt.ylabel("-log10 adjusted p-value")
plt.title("Tumour vs Immune DEGs")
plt.tight_layout()
plt.show()


# Ranked gene list (bar plot) (Tumour vs Immune DEGs)
top_ranked = (
    df.query("pvals_adj < 0.05")
    .sort_values("logfoldchanges", ascending=False)
    .head(20)
)

plt.figure(figsize=(6, 5))
plt.barh(
    top_ranked["names"],
    top_ranked["logfoldchanges"],
    color="firebrick"
)
plt.gca().invert_yaxis()
plt.xlabel("log2 fold change (Tumour vs Immune)")
plt.title("Top tumour-upregulated genes")
plt.tight_layout()
plt.show()


# UMAP feature plots of top tumour-upregulated genes
top_genes = (
    df.query("pvals_adj < 0.05")
    .sort_values("logfoldchanges", ascending=False)
    .head(12)["names"]
    .tolist()    
)

sc.pl.umap(
    adata_ti,
    color=top_genes,
    ncols=4,
    vmax="p99",
    frameon=False,
    title=top_genes
)

# Heatmap of tumour vs immune DEGs
heatmap_genes = (
    tumour_vs_immune
    .query("pvals_adj < 0.05")
    .sort_values("logfoldchanges", ascending=False)
    .head(25)["names"]
    .tolist()
)

sc.pl.heatmap(
    adata_ti,
    var_names=heatmap_genes,
    groupby="broad_type",
    standard_scale="var",
    show_gene_labels=True,
    cmap="viridis"
)


# Dot plot (Tumour vs Immune DEGs)
sc.pl.dotplot(
    adata_ti,
    var_names={
        "Tumour-enriched genes": heatmap_genes
    },
    groupby="broad_type",
    standard_scale="var",
    color_map="Reds"
)



In [ ]:
# Tumour vs Normal DEG

tumour = adata[
    adata.obs["broad_cell_type"] == "Tumour epithelial"
].copy()

tumour.obs["condition"] = (
    tumour.obs["condition"]
    .astype("category")
    .cat.remove_unused_categories()
)

# Sanity check
print(tumour.obs["condition"].value_counts())

sc.tl.rank_genes_groups(
    tumour,
    groupby="condition",
    reference="Normal",
    method="wilcoxon"
)

tumour_cancer_vs_normal = sc.get.rank_genes_groups_df(
    tumour,
    group="Cancer"
)

tumour_cancer_vs_normal.to_csv(
    "GSE184880_tumour_Cancer_vs_Normal_DEGs.csv",
    index=False
)

sig_tumour_genes = tumour_cancer_vs_normal.query(
    "pvals_adj < 0.05 & logfoldchanges > 0.5"
)

up_genes = sig_tumour_genes.sort_values(
    "logfoldchanges", ascending=False
).head(10)["names"].tolist()

down_genes = sig_tumour_genes.sort_values(
    "logfoldchanges", ascending=True
).head(10)["names"].tolist()

sc.pl.umap(
    tumour,
    color=up_genes,
    ncols=5,
    vmax="p99"
)

sc.pl.umap(
    tumour,
    color=down_genes,
    ncols=5,
    vmax="p99"
)

sc.pl.dotplot(
    tumour,
    var_names={
        "Upregulated in Cancer": up_genes,
        "Downregulated in Cancer": down_genes
    },
    groupby="condition"
)

sc.pl.heatmap(
    tumour,
    var_names=up_genes + down_genes,
    groupby="condition",
    standard_scale="var",
    show_gene_labels=True
)



## 6. Immune Heterogeneity & Functional States

The immune compartment (T cells, B cells, NK cells, Macrophages) is re-clustered independently at higher resolution to resolve finer immune subtypes that may be obscured in the whole-dataset clustering. Cells are classified as cycling or non-cycling based on their G2/M score (top 10th percentile threshold), and DEGs between these states are identified. Feature plots of lineage and functional state markers are used to validate subcluster identities.

In [ ]:
# Immune functional states: Cycling vs non-cycling immune cells

immune = adata[
    adata.obs["broad_cell_type"].isin(
        ["T cells", "B cells", "NK cells", "Macrophages"]
    )
].copy()

# Ensure score exists
assert "G2M_score" in immune.obs.columns

threshold = immune.obs["G2M_score"].quantile(0.9)

immune.obs["cycling_state"] = np.where(
    immune.obs["G2M_score"] > threshold,
    "Cycling",
    "Non-cycling"
)

immune.obs["cycling_state"] = (
    immune.obs["cycling_state"]
    .astype("category")
    .cat.remove_unused_categories()
)

immune.obs["cycling_state"].value_counts()


sc.tl.rank_genes_groups(
    immune,
    groupby="cycling_state",
    reference="Non-cycling",
    method="wilcoxon"
)

cycling_degs = sc.get.rank_genes_groups_df(
    immune,
    group="Cycling"
)

cycling_degs.head(10)
cycling_degs.head(20)[["names", "logfoldchanges"]]


cycling_degs.to_csv(
    "GSE184880_immune_cycling_vs_noncycling_DEGs.csv",
    index=False
)

sc.pl.umap(
    immune,
    color=["G2M_score", "cycling_state"],
    frameon=False
)


In [ ]:
# Immune-only subclustering

immune = adata[
    adata.obs["broad_cell_type"].isin(
        ["T cells", "B cells", "NK cells", "Macrophages"]
    )
].copy()

immune.obs["broad_cell_type"].value_counts()

sc.pp.neighbors(immune, n_neighbors=15)
sc.tl.leiden(immune, resolution=0.3, key_added="immune_leiden")
sc.tl.umap(immune)

sc.tl.rank_genes_groups(
    immune,
    groupby="immune_leiden",
    method="wilcoxon"
)

immune_markers = sc.get.rank_genes_groups_df(immune, None)
immune_markers.groupby("group").head(10)

immune_cluster_labels = {
    "0": "CD8 T cells",
    "1": "Macrophages",
    "2": "B cells",
    "3": "NK cells",
    "4": "Naïve / Memory T",
    "5": "FCGR3A+ Monocytes",
    "6": "Cycling immune",
    "7": "Dendritic cells",
    "8": "IFN-stimulated immune"
}

immune.obs["immune_cell_type"] = (
    immune.obs["immune_leiden"]
    .astype(str)
    .map(immune_cluster_labels)
    .astype("category")
)


sc.pl.umap(
    immune,
    color="immune_cell_type",
    title="Immune subclusters (annotated)",
    frameon=False
)

sc.pl.umap(
    immune,
    color="immune_leiden",
    title="Immune subclusters (Leiden)",
    frameon=False
)



immune_subcul = immune.copy()
immune_subcul.write("adata_immune_subclustered.h5ad")

In [ ]:
# Rare immune population deep dive

sc.tl.rank_genes_groups(
    immune,
    groupby="immune_leiden",
    method="wilcoxon"
)

immune_markers = sc.get.rank_genes_groups_df(immune, None)
immune_markers.to_csv(
    "immune_subcluster_markers.csv",
    index=False
)



# UMAP of immune subclusters
sc.pl.umap(
    immune,
    color="immune_leiden",
    title="Immune subclusters (Leiden)"
)

# Dot plot of top markers per immune subcluster
sc.pl.rank_genes_groups_dotplot(
    immune,
    groupby="immune_leiden",
    n_genes=5,
    standard_scale="var",
    title="Dot plot of top markers per immune subcluster"
)

# Heatmap of immune subcluster markers
sc.pl.rank_genes_groups_heatmap(
    immune,
    groupby="immune_leiden",
    n_genes=10,
    standard_scale="var",
    show_gene_labels=True,
)

In [ ]:
# Feature plots of lineage & rare-state markers
# (A) Canonical immune lineage markers
sc.pl.umap(
    immune,
    color=[
        "CD3D", "TRBC1",   # T cells
        "MS4A1",           # B cells
        "LST1", "C1QC",    # Myeloid
        "NKG7", "GNLY"     # NK
    ],
    ncols=3
)

In [ ]:
# Rare / functional state markers
sc.pl.umap(
    immune,
    color=[
        "FCGR3A",   # non-classical monocytes
        "CLEC9A",   # dendritic cells
        "CCR7",     # naïve T
        "IL7R",     # memory T
        "PDCD1",    # exhausted T
        "MKI67"     # cycling immune
    ],
    ncols=3
)

## 7. Immune Functional Interpretation

Three functional gene signatures are scored across all cells to map immune activity in the tumour microenvironment:

- **Exhaustion:** PDCD1, CTLA4, LAG3, TIGIT — markers of T cell dysfunction commonly upregulated in chronic antigen exposure
- **Cytotoxicity:** GZMB, PRF1, NKG7 — effector molecules associated with active cytotoxic killing
- **Interferon response:** IFIT1, IFIT3, ISG15 — interferon-stimulated genes indicative of innate immune activation

Immune checkpoint receptor expression (PD-1, PD-L1, CTLA-4, LAG-3, TIGIT, TIM-3) is visualised across immune subclusters to identify populations most likely to be responsive to checkpoint blockade therapy.

In [ ]:
# Functional state scoring (activation, exhaustion, cytotoxicity)

exhaustion_genes = ["PDCD1", "CTLA4", "LAG3", "TIGIT"]
cytotoxic_genes = ["GZMB", "PRF1", "NKG7"]
interferon_genes = ["IFIT1", "IFIT3", "ISG15"]

sc.tl.score_genes(adata, exhaustion_genes, score_name="Exhaustion")
sc.tl.score_genes(adata, cytotoxic_genes, score_name="Cytotoxicity")
sc.tl.score_genes(adata, interferon_genes, score_name="Interferon")

sc.pl.umap(
    adata,
    color=["Exhaustion", "Cytotoxicity", "Interferon"],
    vmax="p99",
    title=["Exhaustion score", "Cytotoxicity score", "Interferon score"]
)

In [ ]:
# Immune checkpoint analysis

checkpoint_genes = [
    "PDCD1", "CD274", "CTLA4",
    "LAG3", "TIGIT", "HAVCR2"
]

sc.pl.umap(
    immune,
    color=checkpoint_genes,
    vmax="p99",
    title=checkpoint_genes
)

sc.pl.violin(
    immune,
    keys=checkpoint_genes,
    groupby="immune_leiden",
    rotation=90
    
)

In [ ]:
# Proliferative vs non-proliferative cell scoring

sc.pl.umap(
    adata,
    color=["S_score", "G2M_score"],
    vmax=2
)

adata.obs["proliferative"] = (
    (adata.obs["S_score"] + adata.obs["G2M_score"]) > 0
)

adata.obs["proliferative"] = adata.obs["proliferative"].map(
    {True: "Proliferative", False: "Non-proliferative"}
).astype("category")

sc.pl.umap(
    adata,
    color="proliferative",
    title="Proliferative vs Non-proliferative cells"
)

## 8. Systems-Level Interpretation (Pathway Enrichment)

Pathway-level interpretation is performed using two complementary approaches.

**Enrichr** is applied to the top 200 tumour-upregulated genes (ranked by log fold change) against the MSigDB Hallmark gene set collection, providing an overrepresentation analysis of broad biological themes.

**Gene Set Enrichment Analysis (GSEA)** via `gseapy.prerank` uses the full ranked gene list (by log fold change, with a small random jitter to break ties) to detect coordinated pathway enrichment beyond a fixed gene list cutoff. GSEA is run against four collections: MSigDB Hallmark, KEGG, Reactome, and GO Biological Process. Results are visualised as NES bar charts and bubble plots (dot size reflects nominal p-value significance).

In [ ]:
# Pathway enrichment

top_tumour_genes = (
    tumour_vs_immune
    .sort_values("logfoldchanges", ascending=False)
    .head(200)["names"]
    .tolist()
)

enr = gp.enrichr(
    gene_list=top_tumour_genes,
    gene_sets="MSigDB_Hallmark_2020",
    organism="human",
    outdir=None,
    cutoff=1.0,        # <- IMPORTANT
    no_plot=True       # <- IMPORTANT
)

enr.results.to_csv(
    "tumour_cancer_enrichment.csv",
    index=False
)

enr.results.head()
# Ranked GSEA (Tumour vs Immune)

# Build ranked gene list (Tumour vs Immune)
np.random.seed(0)

ranks = (
    tumour_vs_immune
    .dropna(subset=["logfoldchanges"])
    .assign(
        rank_score=lambda x: x["logfoldchanges"] + np.random.normal(0, 1e-6, size=len(x))
    )
    .set_index("names")["rank_score"]
    .sort_values(ascending=False)
)


# Optional sanity check
ranks.head()

gsea_hallmark = gp.prerank(
    rnk=ranks,
    gene_sets="MSigDB_Hallmark_2020",
    organism="human",
    permutation_num=100,      # sufficient for scRNA-seq
    min_size=15,
    max_size=500,
    outdir="GSEA_Tumour_vs_Immune_Hallmark",
    seed=0,
    verbose=True
)

hallmark_results = gsea_hallmark.res2d.reset_index()
hallmark_results.to_csv(
    "tumour_vs_immune_GSEA_Hallmark.csv",
    index=False
)

hallmark_results.head()

gene_sets = {
    "KEGG": "KEGG_2021_Human",
    "Reactome": "Reactome_2022",
    "GO_BP": "GO_Biological_Process_2021"
}

gsea_results = {}

for name, gs in gene_sets.items():
    gsea = gp.prerank(
        rnk=ranks,
        gene_sets=gs,
        organism="human",
        permutation_num=100,
        min_size=15,
        max_size=500,
        outdir=f"GSEA_Tumour_vs_Immune_{name}",
        seed=0,
        verbose=False
    )
    
    res = gsea.res2d.reset_index()
    res.to_csv(f"tumour_vs_immune_GSEA_{name}.csv", index=False)
    gsea_results[name] = res

sig_hallmark = hallmark_results[
    hallmark_results["FDR q-val"] < 0.25
]

sig_hallmark[
    ["Term", "NES", "FDR q-val"]
].sort_values("NES", ascending=False).head(10)


In [ ]:
# 1. Enrichr bar chart – top Hallmark pathways by adjusted p-value
top_enr = (
    enr.results
    .sort_values("Adjusted P-value")
    .head(15)
)

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(top_enr["Term"], -np.log10(top_enr["Adjusted P-value"]), color="steelblue")
ax.axvline(-np.log10(0.05), linestyle="--", color="black", label="p=0.05")
ax.invert_yaxis()
ax.set_xlabel("-log10 adjusted p-value")
ax.set_title("Enrichr – Top Hallmark pathways (Tumour-upregulated genes)")
ax.legend()
plt.tight_layout()
plt.show()

# Show all 50 pathways ranked by NES, coloured by nominal p-value
plot_df = hallmark_results.sort_values("NES", ascending=False)
colors = ["crimson" if n > 0 else "steelblue" for n in plot_df["NES"]]

fig, ax = plt.subplots(figsize=(8, 10))
ax.barh(plot_df["Term"], plot_df["NES"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.invert_yaxis()
ax.set_xlabel("Normalised Enrichment Score (NES)")
ax.set_title("GSEA Hallmark – Tumour vs Immune\n(all pathways, ranked by NES)")
plt.tight_layout()
plt.show()

# Top 10 up and down by NES with nominal p-value as dot size
top = pd.concat([
    plot_df.head(10),
    plot_df.tail(10)
])
top["neg_log_nom_p"] = -np.log10(top["NOM p-val"] + 1e-10)
colors2 = ["crimson" if n > 0 else "steelblue" for n in top["NES"]]

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(top["NES"], top["Term"], 
           c=colors2, s=top["neg_log_nom_p"] * 30,
           alpha=0.8, edgecolors="grey", linewidths=0.5)
ax.axvline(0, linestyle="--", color="grey")
ax.set_xlabel("NES")
ax.set_title("Top/bottom 10 Hallmark pathways\n(dot size = -log10 nominal p-value)")
plt.tight_layout()
plt.show()

# 2. GSEA NES bar chart – significant Hallmark pathways
sig = hallmark_results[hallmark_results["FDR q-val"] < 0.25].copy()
sig = sig.sort_values("NES", ascending=False)

colors = ["crimson" if n > 0 else "steelblue" for n in sig["NES"]]

fig, ax = plt.subplots(figsize=(7, max(4, len(sig) * 0.35)))
ax.barh(sig["Term"], sig["NES"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.invert_yaxis()
ax.set_xlabel("Normalised Enrichment Score (NES)")
ax.set_title("GSEA Hallmark – Tumour vs Immune (FDR < 0.25)")
plt.tight_layout()
plt.show()

# 3. Dot plot – NES vs -log10(FDR) bubble chart
plot_df = hallmark_results[hallmark_results["FDR q-val"] < 0.25].copy()
plot_df["neg_log_fdr"] = -np.log10(plot_df["FDR q-val"] + 1e-10)
plot_df["color"] = plot_df["NES"].apply(lambda x: "crimson" if x > 0 else "steelblue")

fig, ax = plt.subplots(figsize=(6, 5))
scatter = ax.scatter(
    plot_df["NES"],
    plot_df["neg_log_fdr"],
    c=plot_df["color"],
    s=plot_df["neg_log_fdr"] * 20,
    alpha=0.8,
    edgecolors="grey",
    linewidths=0.5
)
for _, row in plot_df.iterrows():
    ax.annotate(row["Term"], (row["NES"], row["neg_log_fdr"]),
                fontsize=7, ha="left", va="bottom")
ax.axvline(0, linestyle="--", color="grey")
ax.set_xlabel("NES")
ax.set_ylabel("-log10 FDR")
ax.set_title("GSEA Hallmark pathway enrichment")
plt.tight_layout()
plt.show()

In [ ]:
# Ranked genes: Cancer vs Normal tumour cells
ranks_tumour = (
    tumour_cancer_vs_normal
    .set_index("names")["logfoldchanges"]
    .sort_values(ascending=False)
)

gsea_tumour = gp.prerank(
    rnk=ranks_tumour,
    gene_sets="MSigDB_Hallmark_2020",
    organism="human",  # <-- fixed
    permutation_num=100,
    outdir="GSEA_Tumour_Cancer_vs_Normal",
    seed=0
)

tumour_gsea_results = gsea_tumour.res2d.reset_index()
tumour_gsea_results.to_csv(
    "tumour_cancer_vs_normal_GSEA_Hallmark.csv",
    index=False
)

print(f"Total pathways: {len(tumour_gsea_results)}")
print(f"Significant (FDR < 0.25): {(tumour_gsea_results['FDR q-val'] < 0.25).sum()}")

# Visualise all pathways ranked by NES
plot_df = tumour_gsea_results.sort_values("NES", ascending=False)
colors = ["crimson" if n > 0 else "steelblue" for n in plot_df["NES"]]

fig, ax = plt.subplots(figsize=(8, 10))
ax.barh(plot_df["Term"], plot_df["NES"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.invert_yaxis()
ax.set_xlabel("Normalised Enrichment Score (NES)")
ax.set_title("GSEA Hallmark – Cancer vs Normal (tumour cells)\n(all pathways, ranked by NES)")
plt.tight_layout()
plt.show()

# Bubble plot – top/bottom 10 by NES
top = pd.concat([plot_df.head(10), plot_df.tail(10)])
top["neg_log_nom_p"] = -np.log10(top["NOM p-val"] + 1e-10)
colors2 = ["crimson" if n > 0 else "steelblue" for n in top["NES"]]

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(top["NES"], top["Term"],
           c=colors2, s=top["neg_log_nom_p"] * 30,
           alpha=0.8, edgecolors="grey", linewidths=0.5)
ax.axvline(0, linestyle="--", color="grey")
ax.set_xlabel("NES")
ax.set_title("Top/bottom 10 Hallmark pathways – Cancer vs Normal\n(dot size = -log10 nominal p-value)")
plt.tight_layout()
plt.show()

## 9. Final Save

All AnnData objects, metadata tables, DEG results, and GSEA outputs are written to disk. Software versions are recorded inside `adata.uns` for reproducibility. Key analysis parameters are saved to `Analysis_Parameters.txt`.

In [ ]:
import sys

print("python:", sys.version)
print("numpy:", np.__version__)
print("scanpy:", sc.__version__)
print("pandas:", pd.__version__)
print("igraph:", igraph.__version__)
print("leidenalg:", leidenalg.__version__)

adata.uns["software_versions"] = {
    "python": sys.version,
    "scanpy": sc.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "igraph": igraph.__version__,
    "leidenalg": leidenalg.__version__
}

summary = {
    "total_cells": adata.n_obs,
    "total_genes_hvg": adata.n_vars,
    "total_genes_raw": adata.raw.n_vars if adata.raw is not None else None
}
print(summary)

adata.obs.to_csv("cell_metadata.csv")
adata.var.to_csv("gene_metadata.csv")
adata.write("GSE184880_scanpy_FULL_ANALYSIS.h5ad")

immune_subclustered = immune.copy()
immune_subclustered.write("GSE184880_scanpy_immune_subclusters.h5ad")

with open("Analysis_Parameters.txt", "w") as f:
    f.write("""
Clustering:
- Leiden resolution: 0.3
- Neighbors: 15
- PCs: 30

QC:
- min genes: 300
- max genes: 6000
- mitochondrial cutoff: 15%

Normalization:
- LogNormalize (1e4)

Cell cycle:
- S and G2M scoring (Tirosh et al.)
- Regression of S_score and G2M_score

HVGs:
- 2000 genes (Seurat flavor, batch-aware)

UMAP:
- Default Scanpy parameters

DEG:
- Test: Wilcoxon
- FDR cutoff: 0.05
""")

print("All outputs saved successfully.")